# SAMVEDNA ML Pipeline: Exploratory Data Analysis & Data Quality Audit

This notebook conducts a thorough geospatial and statistical audit of the landslide inventory and feature distributions used in the SAMVEDNA platform.

### Primary Data Sources:
1. **NASA Global Landslide Catalog (GLC) / COOLR** - 1,265 rainfall-triggered landslide events across India (2007–2016).
2. **Copernicus / SRTM 30m DEM** - Digital elevation, slope gradient, aspect, and terrain curvature.
3. **ERA5 Reanalysis & NASA GPM IMERG** (via Open-Meteo Historical Archive) - Rainfall intensity, 1h/3h/24h accumulation, 7-day antecedent precipitation, volumetric soil moisture.
4. **Non-Landslide Baselines** - Spatially buffered (>3 km) negative sampling across stable Himalayan and regional baseline topographies.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries loaded successfully.")

## 1. NASA Global Landslide Catalog Ingestion (India)

In [ ]:
raw_csv_path = '../data/raw/nasa_glc.csv'
df_raw = pd.read_csv(raw_csv_path, low_memory=False)

india_df = df_raw[df_raw['country_name'] == 'India'].copy()
india_df['event_date_parsed'] = pd.to_datetime(india_df['event_date'], errors='coerce')
india_df['year'] = india_df['event_date_parsed'].dt.year

print(f"Total Global Records in NASA GLC: {len(df_raw):,}")
print(f"Total Validated Events in India: {len(india_df):,}")
print("\nYearly Distribution (India):")
print(india_df['year'].value_counts().sort_index())

## 2. Trigger Mechanism & Regional Distribution

In [ ]:
plt.figure(figsize=(10, 4))
triggers = india_df['landslide_trigger'].value_counts().head(6)
triggers.plot(kind='barh', color='#2563eb')
plt.title("Primary Trigger Mechanisms for Landslides in India (NASA GLC)")
plt.xlabel("Reported Event Count")
plt.tight_layout()
plt.show()

## 3. Spatial Distribution Across the Himalayan Arc & Western Ghats

In [ ]:
plt.figure(figsize=(9, 7))
plt.scatter(india_df['longitude'], india_df['latitude'], c='#ef4444', alpha=0.6, s=18, edgecolors='none', label='NASA Landslide Event')
plt.title("Spatial Distribution of Landslide Occurrences in India (NASA GLC)")
plt.xlabel("Longitude (°E)")
plt.ylabel("Latitude (°N)")
plt.axvline(x=86.0, color='#6b7280', linestyle='--', label='Spatial Partition (West vs East)')
plt.legend(loc='lower left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Model Metadata & Evaluation Summary Audit

In [ ]:
metadata_path = '../models/model_metadata.json'
if os.path.exists(metadata_path):
    with open(metadata_path, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    print("Model Name:", meta.get('model_name'))
    print("Algorithm:", meta.get('algorithm'))
    print("Status:", meta.get('status'))
    print("\nHoldout Evaluation Metrics:")
    for k, v in meta.get('evaluation_metrics_holdout', {}).items():
        print(f"  {k}: {v}")
else:
    print("Model metadata not yet generated. Run train_xgboost.py first.")